# ReGreen — Model Eğitimi

Yangın sonrası rehabilitasyon önceliklendirmesi için model kurma defteri.

**Hedef:** `kalan_acik` = yangın öncesi NDVI − 2. yıl NDVI
Yani *iki yıl sonra bitki örtüsünün ne kadarı hâlâ eksik.*

**Veri:** 16.074 etiketli hücre · 34 yangın · 27 mekânsal grup · **7 öznitelik**

---

## Bu defterde ne var

| Bölüm | Ne yapıyor | Süre |
|---|---|---|
| 1 | Kurulum ve veri yükleme | 1 dk |
| 2 | Ortak altyapı — metrikler ve LOGO | anında |
| 3 | Veriye bakış | anında |
| 4 | **Aşama 1** — model ailesi karşılaştırması | 10–20 dk |
| 5 | **Aşama 2** — hiperparametre araması (iç içe CV) | 15–40 dk |
| 6 | **Aşama 3** — final model + kayıt | 2 dk |
| 7 | Grafikler | 1 dk |

> **Colab ayarı:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → CPU` yeterli.
> GPU'ya gerek yok, ağaç tabanlı modeller CPU'da çalışır.

> **Hücreleri sırayla çalıştır.** Aşama 2, Aşama 1'in sonucunu kullanıyor.

---
# 1. Kurulum ve veri

`egitim_seti.csv` dosyasını yükle. Colab'ın kendi dosya seçicisi açılacak.

In [ ]:
import sys, subprocess
for p in ["lightgbm", "xgboost"]:
    try:
        __import__(p)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=True)

import numpy as np, pandas as pd, time, json
from scipy.stats import spearmanr
print("hazir | pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# egitim_seti.csv dosyasini yukle
try:
    from google.colab import files
    yuklenen = files.upload()
    DOSYA = list(yuklenen.keys())[0]
except ImportError:
    DOSYA = "egitim_seti.csv"      # yerelde calistiriyorsan

d = pd.read_csv(DOSYA)
print(DOSYA, "->", len(d), "satir,", d.shape[1], "sutun")

---
# 2. Ortak altyapı

## Neden hazır `cross_val_score` kullanmıyoruz?

İki sebep var:

**1. Metriğimiz grup bazında.** Grupların hedef ortalamaları 0,123 ile 0,405
arasında değişiyor — üç kat fark. Bütün satırları tek torbaya atıp korelasyon
ölçersek model aslında *"hangi yangın bu"* sorusunu çözmüş olur, *"bu yangının
içinde neresi kötü"* sorusunu değil. Karar vericinin sorduğu ikincisi.

**2. Sıralama yapıyoruz, değer tahmin etmiyoruz.** "Bu hücrenin açığı 0,31 mi
0,34 mü" kimseyi ilgilendirmiyor; "hangi 500 hektardan başlayayım"
ilgilendiriyor. O yüzden **Spearman** (sıra korelasyonu) kullanıyoruz, MSE değil.

## LOGO — Leave One Group Out

Her seferinde bir mekânsal grubu tamamen dışarı çıkarıyoruz:

```
Sınav  1 : Manavgat çıkar → kalan 26 grupla eğit → Manavgat'ı sor
Sınav  2 : Marmaris çıkar → kalan 26 grupla eğit → Marmaris'i sor
...
Sınav 27
```

Gerçek hayatta olacak şey tam olarak bu: yeni yangın çıktığında model orayı
hiç görmemiş olacak. LOGO bunu ölçüyor.

Raporladığımız *"27 grubun 26'sında pozitif"* cümlesi doğrudan bu 27 sınavın
sonucu.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold

OZNITELIK = ["agac_orani", "agac_orani_y", "dnbr", "egim_derece",
             "yukselti_m", "ndvi_dusus", "yol_mesafe_km"]
HEDEF = "kalan_acik"
GRUP  = "grup_id"
TOHUM = 0


# --------------------------------------------------------------- oznitelik
def X_hazirla(d, nan="yerel"):
    # nan = 'yerel'         -> NaN birak (HistGB / LightGBM / XGBoost bunu destekler)
    #       'doldur'        -> RF icin; RF NaN kabul etmez
    #       'doldur_bayrak' -> doldur + "burasi dolduruldu" bilgisini sutun olarak ekle
    X = d[OZNITELIK].copy()
    if nan == "yerel":
        return X
    bayrak = X["agac_orani_y"].isna().astype("int8")
    X["agac_orani_y"] = X["agac_orani_y"].fillna(X["agac_orani"])
    X["yukselti_m"]   = X["yukselti_m"].fillna(X["yukselti_m"].median())
    if nan == "doldur_bayrak":
        X["agac_orani_y_eksik"] = bayrak
    return X


# ----------------------------------------------------------------- agirlik
def agirlik_hesapla(gruplar, sema="yok"):
    # Kat basina yeniden hesaplanir: bir grup disarida kalinca kalanlarin
    # payi degisir, global agirligi tasimak yanlis olur.
    if sema == "yok":
        return None
    s = pd.Series(gruplar)
    ham = len(s) / (s.nunique() * s.map(s.value_counts()))
    a = {"ham": ham, "kok": np.sqrt(ham), "tavan": ham.clip(upper=3.0)}[sema]
    return (a / a.mean()).to_numpy()


# ------------------------------------------------------------------ metrik
def top_isabet(y, p, oran=0.20):
    # Modelin sectigi "en kotu %20", gercekte en kotu %20 icinde mi?
    n = len(y)
    k = max(1, int(round(n * oran)))
    if k >= n:
        return np.nan
    gercek = set(np.argsort(-np.asarray(y))[:k])
    return sum(1 for i in np.argsort(-np.asarray(p))[:k] if i in gercek) / k


def grup_metrik(y, p, gruplar, taban_p=None):
    df = pd.DataFrame({"y": y, "p": p, "g": gruplar})
    if taban_p is not None:
        df["t"] = taban_p
    satir = []
    for gr, s in df.groupby("g"):
        if len(s) < 8 or s["y"].nunique() < 3:      # cok kucuk grupta rho anlamsiz
            continue
        r = {"grup": gr, "n": len(s),
             "rho": spearmanr(s["y"], s["p"]).statistic,
             "top20": top_isabet(s["y"].values, s["p"].values)}
        if taban_p is not None:
            r["top20_taban"] = top_isabet(s["y"].values, s["t"].values)
        satir.append(r)
    return pd.DataFrame(satir)


def ozetle(gm):
    p = gm["rho"].dropna()
    o = {"grup_sayisi": len(gm), "rho_ort": p.mean(), "rho_medyan": p.median(),
         "rho_pozitif": int((p > 0).sum()), "rho_min": p.min(),
         "rho_maks": p.max(), "top20": gm["top20"].mean()}
    if "top20_taban" in gm:
        o["top20_taban"] = gm["top20_taban"].mean()
    return o


# ---------------------------------------------------------------------- CV
def logo_tahmin(kur, X, y, gruplar, agirlik_sema="yok"):
    # LeaveOneGroupOut ile out-of-fold tahmin uretir.
    oof = np.full(len(y), np.nan)
    for tr, te in LeaveOneGroupOut().split(X, y, gruplar):
        m = kur()
        w = agirlik_hesapla(gruplar[tr], agirlik_sema)
        if w is None:
            m.fit(X.iloc[tr], y[tr])
        else:
            m.fit(X.iloc[tr], y[tr], sample_weight=w)
        oof[te] = m.predict(X.iloc[te])
    return oof


def satir_yaz(ad, o):
    print("%-24s %+7.3f %+7.3f %3d/%-3d %+7.3f %6.1f%%" %
          (ad, o["rho_ort"], o["rho_medyan"], o["rho_pozitif"],
           o["grup_sayisi"], o["rho_min"], 100 * o["top20"]))


BASLIK = ("%-24s %7s %7s %7s %7s %7s" %
          ("model", "rho_ort", "rho_med", "poz", "en_kotu", "top20"))
print("altyapi hazir")

---
# 3. Veriye bakış

Eğitmeden önce neyle uğraştığımızı görelim. Özellikle **grup dengesizliğine**
dikkat et — modelin en büyük zorluğu bu. Tek bir grup (Manavgat) verinin
%37'si.

In [ ]:
y = d[HEDEF].to_numpy()
g = d[GRUP].to_numpy()
taban_dnbr = d["dnbr"].to_numpy()      # sahadaki mevcut yontem: "en cok yanan yere git"

print("satir %d | yangin %d | grup %d\n" %
      (len(d), d.yangin_id.nunique(), d[GRUP].nunique()))

print("HEDEF (kalan_acik)")
print(d[HEDEF].describe()[["mean", "std", "min", "max"]].to_string(), "\n")

gb = (d.groupby(GRUP).agg(n=(HEDEF, "size"), ort=(HEDEF, "mean"))
        .sort_values("n", ascending=False))
print("GRUP DAGILIMI")
print(gb.to_string())
print("\nen buyuk grup : %.1f%%   |   ilk 3 grup : %.1f%%" %
      (100 * gb.n.iloc[0] / len(d), 100 * gb.n.iloc[:3].sum() / len(d)))
print("grup ortalamalari: %.3f - %.3f  (gruplar arasi kayma var)" %
      (gb.ort.min(), gb.ort.max()))

print("\nEKSIK DEGER")
eksik = d[OZNITELIK].isna().sum()
print(eksik[eksik > 0].to_string() if eksik.sum() else "yok")

---
# 3b. Öznitelik seti neden bu 7?

Bunu bilmen lazım — jüri en çok buradan soru sorar.

## İçeride olanlar

| Öznitelik | Ne anlatıyor |
|---|---|
| `agac_orani` | Bölge normalde ne kadar ormanlık (WorldCover, çok yıllık) |
| `agac_orani_y` | Yangın yılındaki ağaç örtüsü (Impact Observatory) |
| `dnbr` | Yangın şiddeti — kızılötesi yanık izi |
| `egim_derece` | Erozyon ve tohum tutunma riski |
| `yukselti_m` | İklim / vejetasyon kuşağı |
| `ndvi_dusus` | **Yangın hemen ardındaki fiili bitki kaybı** |
| `yol_mesafe_km` | Müdahale edilebilirlik |

`ndvi_dusus` sona eklendi ve **tek başına +0,086 kazandırdı** — bütün model
aileleri arasındaki farkın iki katı.

> **"dNBR zaten şiddeti ölçmüyor mu?"** Ölçüyor ama farklı bir şeyi: dNBR
> kızılötesi bandındaki *yanık izini* ölçer, `ndvi_dusus` ise *fiilen kaybolan
> bitkiyi*. İkisi birbirinin yerine geçmiyor, üst üste bilgi katıyorlar.

## Reddedilen bir tuzak: `ndvi_oncesi`

Eklendiğinde skoru daha da yükseltiyordu (+0,103). Kullanmadık:

```
ndvi_oncesi tek başına   +0,679
modelle birlikte         +0,675
model katkısı            −0,004   ✗
```

Model hiçbir şey katmıyor. Sebebi matematiksel: hedef
`ndvi_oncesi − ndvi_yil2`, yani özniteliği hedefin içinden veriyoruz.
*"Önce çok bitki vardı, o yüzden açık büyük"* bir tahmin değil, aritmetik.

Kıyasla `ndvi_dusus` gerçek: tek başına +0,594, modelle +0,659, **model
katkısı +0,065.**

> Bu ayrım projenin metodolojik olgunluğunun kanıtı — skoru yükselten ama
> anlamsız olan bir özniteliği ölçüp reddettik.

## Sızıntı olmadığının kanıtı

Tek ölçüt şu: **yangından ~2 hafta sonra bu değer elimizde olur mu?**

- `ndvi_oncesi` ve `ndvi_sonrasi` → evet, ikisi de uydudan geliyor ✓
- `ndvi_yil2` → hayır, o zaten tahmin ettiğimiz şey ✗
- 2 yıllık yağış → hayır, geleceği bilmiyoruz ✗

`ndvi_dusus` ilk gruptan, o yüzden meşru.

## Ölçülüp elenenler

| Aday | Sonuç | Sebep |
|---|---|---|
| Bakı (aspect) | +0,006 / −0,005 | Gürültü. sin/cos dönüşümü de kurtarmadı |
| Su mesafesi | +0,006 | Gürültü |
| Yerleşim mesafesi | −0,011 | Gürültü, yol mesafesiyle örtüşüyor |
| Uzun dönem yağış | **−0,110** | Yangın içinde neredeyse sabit (0,076) — grup içi metrikte bilgi taşımıyor, sadece gürültü |

---
# 4. Aşama 1 — Model ailesi karşılaştırması

Burada **ayar yapmıyoruz**. Amaç hangi ailenin potansiyeli var onu görmek,
en iyi skoru bulmak değil. Hepsi makul varsayılanlarla, hepsi aynı LOGO ile.

Karşılaştırdıklarımız:

| Aday | Neden listede |
|---|---|
| **RF taban** | Şu anki modelimiz — referans noktası |
| **RF + eksik bayrağı** | `agac_orani_y` 701 satırda eksik. Doldurmak yerine "burası doldurulmuş" bilgisini de modele veriyoruz |
| **RF + kök ağırlık** | Manavgat verinin %37'si. Grupları dengelemeyi deniyoruz |
| **HistGB / LightGBM / XGBoost** | NaN'ı yerel destekliyorlar, doldurma varsayımına hiç gerek kalmıyor |
| **Ridge** | Doğrusal model. Sağlama için: ağaçlar bundan iyi değilse problemde doğrusal olmayan yapı yok demektir |
| **dNBR** | Sahada şu an kullanılan "en çok yanan yere git" mantığı. **Yenmesi gereken taban** |

### Eski veriyle (6 öznitelik) çıkan sonuç

Bu adaylar bir kez koşturuldu, bilinmesi gerekenler:

- **8 aile 0,04 aralığına sıkıştı**, doğrusal Ridge en iyiden sadece 0,03
  geride. Yani tavanı model değil veri koyuyor — `ndvi_dusus` bu yüzden eklendi.
- **Ağırlıklandırma zarar veriyor:** kök +0,553, ham +0,537, ağırlıksız +0,568.
  Küçük gruplar (33–70 satır) gürültülü, ağırlık verince model gürültü
  öğreniyor. Ham ağırlık listeden çıkarıldı, kök doğrulama için duruyor.
- **Eksik bayrağı ucuz kazanç:** en kötü grubu −0,026'dan +0,092'ye çıkarıp
  27/27 yapmıştı.

Şimdi 7 öznitelikle tekrar koşuyoruz — sıralama değişiyor mu görelim.

> ⏱️ RF varyantları Colab'da yavaş (27 kat × 400 ağaç, tek çekirdek — yaklaşık
> 10 dk/varyant). Hızlı geçmek istersen `HIZLI = True` yap; ağaç sayısı 150'ye
> iner, sonuç biraz oynar ama sıralama değişmez.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

HIZLI = False                # True -> RF hizlanir
NA = 150 if HIZLI else 400


def RF(**k):
    v = dict(n_estimators=NA, min_samples_leaf=5, random_state=TOHUM, n_jobs=-1)
    v.update(k)
    return RandomForestRegressor(**v)


DENEY = [
    ("RF taban (mevcut)",  "doldur",        "yok", RF),
    ("RF + eksik bayragi", "doldur_bayrak", "yok", RF),
    ("RF + kok agirlik",   "doldur",        "kok", RF),
    ("HistGB",             "yerel",         "yok",
     lambda: HistGradientBoostingRegressor(random_state=TOHUM)),
    ("LightGBM",           "yerel",         "yok",
     lambda: LGBMRegressor(random_state=TOHUM, n_jobs=-1, verbose=-1)),
    ("XGBoost",            "yerel",         "yok",
     lambda: XGBRegressor(random_state=TOHUM, n_jobs=-1, tree_method="hist")),
    ("Ridge (dogrusal)",   "doldur",        "yok",
     lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0))),
]

sonuc, oof_saklanan = [], {}
print("LeaveOneGroupOut - %d grup, %d satir\n" % (d[GRUP].nunique(), len(d)))
print(BASLIK)
print("-" * 64)

for ad, nan, ag, kur_f in DENEY:
    t0 = time.time()
    oof = logo_tahmin(kur_f, X_hazirla(d, nan), y, g, ag)
    o = ozetle(grup_metrik(y, oof, g, taban_dnbr))
    o.update({"ad": ad, "nan": nan, "agirlik": ag,
              "saniye": round(time.time() - t0, 1)})
    sonuc.append(o)
    oof_saklanan[ad] = oof
    satir_yaz(ad, o)

print("-" * 64)
satir_yaz("dNBR (SAHA TABANI)", ozetle(grup_metrik(y, taban_dnbr, g)))

asama1 = pd.DataFrame(sonuc).sort_values("rho_ort", ascending=False)
print("\n\nSIRALAMA")
print(asama1[["ad", "rho_ort", "rho_medyan", "rho_pozitif",
              "rho_min", "top20", "saniye"]].to_string(index=False))

### Tablo nasıl okunur

| Sütun | Anlamı | İyisi |
|---|---|---|
| `rho_ort` | 27 grubun Spearman ortalaması | yüksek |
| `rho_med` | medyanı — birkaç kötü grup ortalamayı çekiyorsa ikisi ayrışır | yüksek |
| `poz` | kaç grupta model doğru yönde çalışıyor | 27/27 |
| `en_kotu` | en kötü grubun skoru — **dayanıklılık göstergesi** | pozitif |
| `top20` | "en kötü %20"yi doğru seçme oranı | yüksek |

**En çok baktığımız iki sütun: `poz` ve `en_kotu`.** Ortalaması yüksek ama bir
grupta ters çalışan model sahada güvenilmez. Sistemin iddiası "her yangında
işe yarıyor" olmalı.

> `rho_ort` farkı **0,01'in altındaysa gürültüdür**, model değişikliği sayma.

---
# 5. Aşama 2 — Hiperparametre araması

## Buradaki tuzak

Parametreleri LOGO üzerinde arayıp sonra yine LOGO skorunu raporlarsak,
o skor **şişer**. Çünkü parametreleri zaten o sınavın cevaplarına bakarak
seçmiş oluruz. 27 grup az, buna çok kolay düşülür.

## Çözüm: iç içe (nested) CV

```
DIŞ döngü : LOGO — 27 grup, sadece RAPOR eder
   içeride: o grup hariç veride GroupKFold(4) ile parametre ARAR
            → en iyi parametreyi bulur
            → o parametreyle eğitir
            → dışarıdaki gruba tahmin verir
```

Dıştaki grup, arama sırasında **hiç görülmüyor**. O yüzden rapor edilen skor
dürüst.

Maliyeti: `27 × n_deneme × 4` eğitim. LightGBM/HistGB hızlı olduğu için
`N_DENEME = 25` ile Colab'da 15–40 dakika sürüyor. RF seçersen daha uzun.

> **Önce Aşama 1'i çalıştır**, kazanan aileyi aşağıya `AILE` olarak yaz.

In [ ]:
def grup_ici_rho(y, p, gruplar):
    # Tek sayi: gruplarin Spearman ortalamasi. Arama bunu maksimize eder.
    df = pd.DataFrame({"y": y, "p": p, "g": gruplar})
    r = []
    for _, s in df.groupby("g"):
        if len(s) < 8 or s["y"].nunique() < 3:
            continue
        v = spearmanr(s["y"], s["p"]).statistic
        if np.isfinite(v):
            r.append(v)
    return float(np.mean(r)) if r else -1.0


def ic_skor(kur_f, params, X, y, gruplar, kat=4):
    kat = min(kat, len(np.unique(gruplar)))
    oof = np.full(len(y), np.nan)
    for tr, te in GroupKFold(n_splits=kat).split(X, y, gruplar):
        m = kur_f(**params)
        m.fit(X.iloc[tr], y[tr])
        oof[te] = m.predict(X.iloc[te])
    return grup_ici_rho(y, oof, gruplar)


def rastgele_ara(kur_f, uzay, X, y, gruplar, n_deneme=25, tohum=0, kat=4):
    rng = np.random.default_rng(tohum)
    en_iyi, en_iyi_skor = None, -np.inf
    for _ in range(n_deneme):
        p = {k: (v[int(rng.integers(len(v)))] if isinstance(v, list) else v(rng))
             for k, v in uzay.items()}
        s = ic_skor(kur_f, p, X, y, gruplar, kat)
        if s > en_iyi_skor:
            en_iyi, en_iyi_skor = p, s
    return en_iyi, en_iyi_skor


print("arama araclari hazir")

In [ ]:
# ---- ASAMA 1'in kazananini buraya yaz ----------------------------------
AILE     = "HistGB"        # "HistGB" | "LightGBM" | "XGBoost" | "RF"
N_DENEME = 25              # arama butcesi; 15 yaparsan hizlanir
# -----------------------------------------------------------------------

if AILE == "LightGBM":
    NAN = "yerel"
    kur = lambda **k: LGBMRegressor(random_state=TOHUM, n_jobs=-1, verbose=-1, **k)
    UZAY = {
        "n_estimators":      [200, 400, 700, 1000],
        "learning_rate":     lambda r: float(10 ** r.uniform(-2.2, -0.7)),
        "num_leaves":        [15, 31, 63, 127],
        "min_child_samples": [10, 20, 40, 80],
        "subsample":         lambda r: float(r.uniform(0.6, 1.0)),
        "subsample_freq":    [1],
        "colsample_bytree":  lambda r: float(r.uniform(0.6, 1.0)),
        "reg_lambda":        lambda r: float(10 ** r.uniform(-2, 1.5)),
    }
elif AILE == "HistGB":
    NAN = "yerel"
    kur = lambda **k: HistGradientBoostingRegressor(random_state=TOHUM, **k)
    UZAY = {
        "max_iter":          [200, 400, 700],
        "learning_rate":     lambda r: float(10 ** r.uniform(-2.2, -0.7)),
        "max_leaf_nodes":    [15, 31, 63],
        "min_samples_leaf":  [10, 20, 40, 80],
        "l2_regularization": lambda r: float(10 ** r.uniform(-3, 1)),
    }
elif AILE == "XGBoost":
    NAN = "yerel"
    kur = lambda **k: XGBRegressor(random_state=TOHUM, n_jobs=-1,
                                   tree_method="hist", **k)
    UZAY = {
        "n_estimators":     [200, 400, 700],
        "learning_rate":    lambda r: float(10 ** r.uniform(-2.2, -0.7)),
        "max_depth":        [3, 4, 6, 8],
        "min_child_weight": [1, 5, 20],
        "subsample":        lambda r: float(r.uniform(0.6, 1.0)),
        "colsample_bytree": lambda r: float(r.uniform(0.6, 1.0)),
        "reg_lambda":       lambda r: float(10 ** r.uniform(-2, 1.5)),
    }
else:                                        # RF
    NAN = "doldur_bayrak"
    kur = lambda **k: RandomForestRegressor(random_state=TOHUM, n_jobs=-1, **k)
    UZAY = {
        "n_estimators":     [300, 600],
        "min_samples_leaf": [1, 3, 5, 10, 20],
        "max_features":     [0.4, 0.6, 0.8, 1.0],
        "max_depth":        [None, 10, 16],
    }

X = X_hazirla(d, NAN)
print("aile:", AILE, "| nan:", NAN)
print("oznitelik:", list(X.columns))

In [ ]:
# --- IC ICE CV --- uzun surer, ilerleme yazdirilir
t0 = time.time()
oof_ayarli = np.full(len(y), np.nan)
secilen = []

for i, (tr, te) in enumerate(LeaveOneGroupOut().split(X, y, g), 1):
    p, s = rastgele_ara(kur, UZAY, X.iloc[tr], y[tr], g[tr], N_DENEME, TOHUM)
    m = kur(**p)
    m.fit(X.iloc[tr], y[tr])
    oof_ayarli[te] = m.predict(X.iloc[te])
    secilen.append({"grup": g[te][0], **p, "ic_rho": round(s, 4)})
    print("  %2d/%d  %-16s ic_rho %+0.3f   (%.0f sn)" %
          (i, d[GRUP].nunique(), g[te][0][:16], s, time.time() - t0))

print("\ntoplam %.1f dk" % ((time.time() - t0) / 60))
ayar_df = pd.DataFrame(secilen)

In [ ]:
o_ayar  = ozetle(grup_metrik(y, oof_ayarli, g, taban_dnbr))
o_taban = ozetle(grup_metrik(y, oof_saklanan["RF taban (mevcut)"], g, taban_dnbr))

print(BASLIK)
print("-" * 64)
satir_yaz("RF taban (mevcut)", o_taban)
satir_yaz(AILE + " ayarli", o_ayar)
print("-" * 64)

print("\nKAZANC")
for k, ad in [("rho_ort", "rho ortalama"), ("rho_medyan", "rho medyan"),
              ("rho_min", "en kotu grup"), ("top20", "top-%20 isabet")]:
    print("  %-16s %+0.4f -> %+0.4f   (%+0.4f)" %
          (ad, o_taban[k], o_ayar[k], o_ayar[k] - o_taban[k]))
print("  %-16s %d/%d -> %d/%d" %
      ("pozitif grup", o_taban["rho_pozitif"], o_taban["grup_sayisi"],
       o_ayar["rho_pozitif"], o_ayar["grup_sayisi"]))

print("\n\n27 KATTA SECILEN PARAMETRELER (kararli mi?)")
print(ayar_df.describe(include="all").T.to_string())

### Karar anı

Kazanç `rho_ort` üzerinde **+0,02'den küçükse** ayar yapmaya değmedi demektir —
tabanla devam et, karmaşıklık eklemenin anlamı yok. Bunu söylemek de bir
sonuçtur: *"aradık, taban zaten iyiydi."*

Ayrıca **27 katta seçilen parametrelere bak.** Her katta bambaşka parametre
seçiliyorsa arama gürültüye uyuyor demektir, güvenme. Parametreler birbirine
benziyorsa gerçek bir optimum var demektir.

---
# 6. Aşama 3 — Final model

Kazanan yapıyı **bütün veriyle** tekrar eğitip kaydediyoruz.

Parametreleri seçmek için iç aramayı bu sefer tüm veri üzerinde bir kez
koşuyoruz. Bu meşru, çünkü **raporladığımız skoru yukarıdaki iç içe CV'den
aldık** — buradaki eğitim sadece üretime gidecek modeli üretmek için.

In [ ]:
KULLAN = "ayarli"       # "ayarli" veya "taban" -- Asama 2'nin sonucuna gore sec

if KULLAN == "ayarli":
    en_iyi_p, en_iyi_s = rastgele_ara(kur, UZAY, X, y, g, N_DENEME, TOHUM)
    print("secilen parametreler (ic rho %+0.3f):" % en_iyi_s)
    for k, v in en_iyi_p.items():
        print("   %-20s %s" % (k, v))
    final = kur(**en_iyi_p)
    final_nan, final_oof = NAN, oof_ayarli
else:
    en_iyi_p = dict(n_estimators=400, min_samples_leaf=5)
    final = RandomForestRegressor(random_state=TOHUM, n_jobs=-1, **en_iyi_p)
    final_nan, final_oof = "doldur_bayrak", oof_saklanan["RF + eksik bayragi"]

Xf = X_hazirla(d, final_nan)
final.fit(Xf, y)
print("\nfinal model egitildi:", type(final).__name__, "|", len(Xf), "satir")

In [ ]:
# --- oznitelik onemi ---
# HistGradientBoosting'in feature_importances_ ozelligi YOK, Ridge'inki de
# bir Pipeline icinde. Ucunu de kaldiran bir yol izliyoruz.
from sklearn.inspection import permutation_importance

if hasattr(final, "feature_importances_"):          # RF, LightGBM, XGBoost
    onem = pd.Series(final.feature_importances_, index=Xf.columns)
    yontem = "dahili (impurity)"
elif hasattr(final, "__getitem__") and hasattr(final[-1], "coef_"):   # Ridge
    onem = pd.Series(np.abs(final[-1].coef_), index=Xf.columns)
    yontem = "katsayi buyuklugu"
else:                                                # HistGB
    pi = permutation_importance(final, Xf, y, n_repeats=10,
                                random_state=TOHUM, n_jobs=-1)
    onem = pd.Series(np.clip(pi.importances_mean, 0, None), index=Xf.columns)
    yontem = "permutasyon (10 tekrar)"

onem = (100 * onem / onem.sum()).sort_values(ascending=False)
print("OZNITELIK ONEMI (%%) - yontem: %s\n" % yontem)
for k, v in onem.items():
    print("  %-24s %5.1f  %s" % (k, v, "#" * int(round(v / 2))))

In [ ]:
# --- kaydet ---
import joblib

joblib.dump({"model": final,
             "oznitelik": list(Xf.columns),
             "nan": final_nan,
             "parametre": en_iyi_p,
             "hedef": HEDEF,
             "olcum": {k: float(v) for k, v in
                       ozetle(grup_metrik(y, final_oof, g)).items()}},
            "regreen_model.joblib")

d_out = d[[GRUP, "yangin_id", "hucre_id", HEDEF]].copy()
d_out["oof_tahmin"] = final_oof
d_out.to_csv("oof_tahminler.csv", index=False)

grup_metrik(y, final_oof, g, taban_dnbr).to_csv("grup_skorlari.csv", index=False)
asama1.to_csv("asama1_aile.csv", index=False)
try:
    ayar_df.to_csv("asama2_parametreler.csv", index=False)
except NameError:
    pass

print("yazildi: regreen_model.joblib, oof_tahminler.csv,")
print("         grup_skorlari.csv, asama1_aile.csv, asama2_parametreler.csv")

---
# 7. Grafikler

Sunumda doğrudan kullanabileceğin üç grafik.

In [ ]:
import matplotlib.pyplot as plt

gm = grup_metrik(y, final_oof, g, taban_dnbr).sort_values("rho")
fig, ax = plt.subplots(1, 3, figsize=(17, 5.4))

# 1) grup bazinda rho
r = ax[0]
r.barh(gm.grup, gm.rho, color=np.where(gm.rho > 0, "#2d6a4f", "#9d0208"))
r.axvline(0, color="k", lw=.8)
r.axvline(gm.rho.mean(), color="#e85d04", ls="--", lw=1.5,
          label="ortalama %+.3f" % gm.rho.mean())
r.set_title("Her mekansal grupta siralama gucu")
r.set_xlabel("Spearman rho")
r.tick_params(labelsize=7)
r.legend(fontsize=8)

# 2) top-%20 karsilastirma
r = ax[1]
deger = [0.20, gm.top20_taban.mean(), gm.top20.mean()]
r.bar(["rastgele", "dNBR\n(saha)", "ReGreen"], deger,
      color=["#adb5bd", "#f4a261", "#2d6a4f"])
for i, v in enumerate(deger):
    r.text(i, v + .01, "%.0f%%" % (100 * v), ha="center", fontweight="bold")
r.set_ylim(0, max(deger) * 1.28)
r.set_title("En kotu %20'yi dogru secme orani")

# 3) tahmin vs gercek
r = ax[2]
r.scatter(final_oof, y, s=3, alpha=.15, color="#1d3557")
lim = [min(final_oof.min(), y.min()), max(final_oof.max(), y.max())]
r.plot(lim, lim, "r--", lw=1)
r.set_xlabel("tahmin")
r.set_ylabel("gercek")
r.set_title("Out-of-fold tahmin vs gercek")

plt.tight_layout()
plt.savefig("model_sonuc.png", dpi=150)
plt.show()

In [ ]:
# --- indir ---
try:
    from google.colab import files
    for f in ["regreen_model.joblib", "oof_tahminler.csv", "grup_skorlari.csv",
              "asama1_aile.csv", "asama2_parametreler.csv", "model_sonuc.png"]:
        try:
            files.download(f)
        except Exception as e:
            print("atlandi:", f, "|", e)
except ImportError:
    print("yerelde calisiyorsun, dosyalar klasorde")

---
# Bittiğinde bana şunları gönder

1. **`asama1_aile.csv`** — hangi aile kazandı
2. **Aşama 2 karşılaştırma çıktısı** — kazanç ne kadar oldu
3. **`grup_skorlari.csv`** — hangi grupta iyi, hangisinde kötü

Buna göre `teslim_uret.py`'yi yeni modelle güncelleyip Beytullah'a giden paketi
yeniden üretiriz. Sunumdaki sayıları (grup içi Spearman, top-%20) da o zaman
tazeleriz.

---

## Yenmesi gereken çıta

Yerelde HistGB ile ölçüldü — LOGO, 27 grup, **7 öznitelikli yeni veri**:

| Metrik | Değer |
|---|---|
| rho ortalama | **+0,659** |
| pozitif grup | **27/27** |
| en kötü grup | **+0,174** |
| top-%20 isabet | **%52,8** |
| dNBR (saha tabanı) | +0,377 · %38,8 |

Aşama 2'nin bunu geçmesi lazım. Geçemezse taban modelle gideriz.

### Nereden geldi bu sıçrama

6 öznitelikli eski veriyle taban **+0,573 / %45,9** idi. Tek fark
`ndvi_dusus` sütunu:

| | rho_ort | poz | en kötü | top20 |
|---|---|---|---|---|
| 6 öznitelik | +0,573 | 27/27 | +0,104 | %45,9 |
| **7 öznitelik** | **+0,659** | **27/27** | **+0,174** | **%52,8** |